# Stage 3: Hierarchiczny model GNN (PyTorch Geometric)

Notatnik zakłada, że etap 2 został uruchomiony i wygenerował artefakty (`stage2_artifacts`).

Pipeline:
1. Wczytanie danych z etapu 2.
2. Budowa grafów molekularnych z `SMILES`.
3. Trening prostego GNN dla 500 klas (multi-label).
4. Użycie hierarchii (`is_a`) jako kary w funkcji kosztu.
5. Ewaluacja globalna i per-poziom (18 poziomów).
6. Zapis checkpointu i predykcji.

In [1]:
from pathlib import Path
import json
import random
import re
from collections import defaultdict
from functools import lru_cache

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINConv, global_mean_pool

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [2]:
# Sciezki
cwd = Path.cwd()
ONTOLOGY_DIR = cwd / "1_ontology" if (cwd / "1_ontology").exists() else cwd

DATA_DIR = ONTOLOGY_DIR / "data"
EXTRAS_DIR = ONTOLOGY_DIR / "extras"
ART_DIR = DATA_DIR / "stage2_artifacts"

NPZ_PATH = ART_DIR / "stage2_fingerprints.npz"
META_PATH = ART_DIR / "stage2_fingerprints_meta.json"
ROW_INDEX_PATH = ART_DIR / "stage2_row_index.parquet"
STAGE1_PATH = DATA_DIR / "chebi_dataset_train_stage1.parquet"
OBO_PATH = EXTRAS_DIR / "chebi_classes.obo"

for p in [NPZ_PATH, META_PATH, ROW_INDEX_PATH, STAGE1_PATH, OBO_PATH]:
    print(p, "OK" if p.exists() else "MISSING")

if not NPZ_PATH.exists():
    raise FileNotFoundError("Brak artefaktow etapu 2. Uruchom stage2_fingerprints_notebook.ipynb.")

c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints.npz OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_fingerprints_meta.json OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage2_artifacts\stage2_row_index.parquet OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\chebi_dataset_train_stage1.parquet OK
c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\extras\chebi_classes.obo OK


In [3]:
# Wczytanie artefaktow etapu 2
npz = np.load(NPZ_PATH)
Y_np = npz["Y"]
train_idx = npz["train_idx"].astype(np.int64)
valid_idx = npz["valid_idx"].astype(np.int64)
M_parent_np = npz["M_parent"].astype(np.uint8)
M_ancestor_np = npz["M_ancestor"].astype(np.uint8)

if META_PATH.exists():
    meta = json.loads(META_PATH.read_text(encoding="utf-8"))
    class_cols = meta.get("class_columns", [f"class_{i}" for i in range(Y_np.shape[1])])
else:
    meta = {}
    class_cols = [f"class_{i}" for i in range(Y_np.shape[1])]

assert Y_np.shape[1] == 500, f"Oczekiwano 500 klas, otrzymano: {Y_np.shape[1]}"
print("Y:", Y_np.shape)
print("train/valid:", len(train_idx), len(valid_idx))
print("M_parent:", M_parent_np.shape, "edges:", int(M_parent_np.sum()))
print("M_ancestor:", M_ancestor_np.shape, "edges:", int(M_ancestor_np.sum()))

Y: (33631, 500)
train/valid: 20682 12949
M_parent: (500, 500) edges: 748
M_ancestor: (500, 500) edges: 8610


In [4]:
# Odczyt SMILES dla tych samych wierszy co w etapie 2
row_index = pd.read_parquet(ROW_INDEX_PATH)
if "smiles_column" in meta:
    smiles_col = meta["smiles_column"]
else:
    smiles_col = "canonical_smiles" if "canonical_smiles" in row_index.columns else "SMILES"

if smiles_col not in row_index.columns:
    # fallback: pobranie ze stage1
    stage1_df = pd.read_parquet(STAGE1_PATH)
    smiles_col = "canonical_smiles" if "canonical_smiles" in stage1_df.columns else "SMILES"
    smiles_series = stage1_df[smiles_col].astype(str).reset_index(drop=True)
else:
    smiles_series = row_index[smiles_col].astype(str).reset_index(drop=True)

assert len(smiles_series) == Y_np.shape[0], "Niezgodna liczba rekordow miedzy stage2 artefaktami a SMILES"
print("SMILES kolumna:", smiles_col)
print("Liczba SMILES:", len(smiles_series))

SMILES kolumna: canonical_smiles
Liczba SMILES: 33631


In [5]:
# Poziomy hierarchii z OBO (powinno byc 18 poziomow)
def parse_obo_parents(path: Path):
    parents = defaultdict(set)
    nodes = set()
    current_id = None
    for raw in path.read_text(encoding="utf-8", errors="replace").splitlines():
        s = raw.strip()
        if s == "[Term]":
            current_id = None
            continue
        if s.startswith("id: "):
            current_id = s[4:].strip()
            nodes.add(current_id)
            parents.setdefault(current_id, set())
            continue
        if s.startswith("is_a: ") and current_id:
            p = s[6:].split("!")[0].strip()
            if p:
                parents[current_id].add(p)
                nodes.add(p)
    return parents, nodes

parents_map, all_nodes = parse_obo_parents(OBO_PATH)

@lru_cache(None)
def depth(node: str) -> int:
    ps = [p for p in parents_map.get(node, set()) if p in all_nodes]
    if not ps:
        return 0
    return 1 + max(depth(p) for p in ps)

class_levels = {c: depth(c) for c in class_cols}
n_levels = max(class_levels.values()) + 1
print("Liczba poziomow:", n_levels)
assert n_levels == 18, f"Oczekiwano 18 poziomow, znaleziono: {n_levels}"

Liczba poziomow: 18


In [6]:
# Cechy atomow i budowa grafow PyG
def atom_features(atom: Chem.Atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization()),
        atom.GetTotalNumHs(),
        int(atom.GetIsAromatic()),
        atom.GetMass() * 0.01,
    ]


def mol_to_data(smiles: str, y_vec: np.ndarray):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_list = []
    for b in mol.GetBonds():
        i = b.GetBeginAtomIdx()
        j = b.GetEndAtomIdx()
        edge_list.append([i, j])
        edge_list.append([j, i])

    if edge_list:
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    y = torch.tensor(y_vec, dtype=torch.float).view(1, -1)
    return Data(x=x, edge_index=edge_index, y=y)


graphs = []
bad_idx = []
for i, (smi, y) in enumerate(zip(smiles_series.tolist(), Y_np)):
    data = mol_to_data(smi, y)
    if data is None:
        bad_idx.append(i)
        continue
    data.sample_idx = i
    graphs.append(data)

print("Grafy OK:", len(graphs), "Bledne:", len(bad_idx))
if len(graphs) == 0:
    raise RuntimeError("Brak poprawnych grafow do treningu.")

[18:41:43] WARNING: not removing hydrogen atom without neighbors
[18:41:43] WARNING: not removing hydrogen atom without neighbors
[18:41:43] WARNING: not removing hydrogen atom without neighbors
[18:41:43] WARNING: not removing hydrogen atom without neighbors
[18:41:43] WARNING: not removing hydrogen atom without neighbors
[18:41:43] WARNING: not removing hydrogen atom without neighbors
[18:41:43] WARNING: not removing hydrogen atom without neighbors
[18:41:43] WARNING: not removing hydrogen atom without neighbors
[18:41:44] WARNING: not removing hydrogen atom without neighbors
[18:41:44] Unusual charge on atom 0 number of radical electrons set to zero
[18:41:44] WARNING: not removing hydrogen atom without neighbors
[18:41:44] WARNING: not removing hydrogen atom without neighbors
[18:41:44] WARNING: not removing hydrogen atom without neighbors
[18:41:44] WARNING: not removing hydrogen atom without neighbors
[18:41:45] WARNING: not removing hydrogen atom without neighbors
[18:41:45] WAR

Grafy OK: 33631 Bledne: 0


In [7]:
# Mapowanie indeksow (po ewentualnym odrzuceniu blednych SMILES)
old_to_new = {}
for new_i, g in enumerate(graphs):
    old_to_new[int(g.sample_idx)] = new_i

train_new = [old_to_new[i] for i in train_idx if i in old_to_new]
valid_new = [old_to_new[i] for i in valid_idx if i in old_to_new]

train_data = [graphs[i] for i in train_new]
valid_data = [graphs[i] for i in valid_new]

print("Train/valid po filtracji:", len(train_data), len(valid_data))

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=128, shuffle=False)

Train/valid po filtracji: 20682 12949


In [8]:
# Model GNN (baseline)
class HierGNN(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 128, out_dim: int = 500, dropout: float = 0.2):
        super().__init__()
        self.mlp1 = nn.Sequential(nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.mlp2 = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, hidden_dim))
        self.conv1 = GINConv(self.mlp1)
        self.conv2 = GINConv(self.mlp2)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.dropout(x)

        x = global_mean_pool(x, batch)
        return self.head(x)


in_dim = train_data[0].x.shape[1]
model = HierGNN(in_dim=in_dim, hidden_dim=128, out_dim=500, dropout=0.2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
bce_loss = nn.BCEWithLogitsLoss()
SOFT_F1_ALPHA = 0.20

parent_child, parent_parent = np.where(M_parent_np == 1)
pc_idx = torch.tensor(parent_child, dtype=torch.long, device=device)
pp_idx = torch.tensor(parent_parent, dtype=torch.long, device=device)

def hierarchy_penalty(logits: torch.Tensor) -> torch.Tensor:
    if pc_idx.numel() == 0:
        return torch.zeros((), device=logits.device)
    probs = torch.sigmoid(logits)
    return torch.relu(probs[:, pc_idx] - probs[:, pp_idx]).mean()


def soft_f1_loss(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    tp = (probs * targets).sum(dim=0)
    fp = (probs * (1 - targets)).sum(dim=0)
    fn = ((1 - probs) * targets).sum(dim=0)
    soft_f1 = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    return 1.0 - soft_f1.mean()

print(model)

HierGNN(
  (mlp1): Sequential(
    (0): Linear(in_features=7, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (mlp2): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (conv1): GINConv(nn=Sequential(
    (0): Linear(in_features=7, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  ))
  (conv2): GINConv(nn=Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  ))
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (head): Linear(in_features=128, out_features=500, bias=True)
)


In [9]:
# Trening + ewaluacja
def predict_logits(model: nn.Module, loader: DataLoader) -> np.ndarray:
    model.eval()
    out = []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            logits = model(batch.x, batch.edge_index, batch.batch)
            out.append(logits.cpu().numpy())
    return np.vstack(out) if out else np.empty((0, 500), dtype=np.float32)


def reshape_batch_targets(y: torch.Tensor, n_classes: int = 500) -> torch.Tensor:
    if y.dim() == 1:
        if y.numel() % n_classes != 0:
            raise ValueError(f"Nie mozna przeksztalcic y o ksztalcie {tuple(y.shape)} do (*, {n_classes})")
        return y.view(-1, n_classes)
    if y.dim() == 2 and y.shape[1] == n_classes:
        return y
    if y.dim() > 2 and y.shape[-1] == n_classes:
        return y.view(-1, n_classes)
    raise ValueError(f"Nieoczekiwany ksztalt y: {tuple(y.shape)}")


def collect_targets(loader: DataLoader) -> np.ndarray:
    ys = []
    for batch in loader:
        yb = reshape_batch_targets(batch.y, n_classes=500)
        ys.append(yb.cpu().numpy())
    return np.vstack(ys) if ys else np.empty((0, 500), dtype=np.float32)


def macro_ap(y_true: np.ndarray, y_score: np.ndarray) -> float:
    aps = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        if np.unique(yt).size < 2:
            continue
        aps.append(average_precision_score(yt, y_score[:, c]))
    return float(np.mean(aps)) if aps else float("nan")


def micro_ap(y_true: np.ndarray, y_score: np.ndarray) -> float:
    if y_true.size == 0:
        return float("nan")
    return float(average_precision_score(y_true.ravel(), y_score.ravel()))


def macro_f1(y_true: np.ndarray, y_pred_bin: np.ndarray) -> float:
    f1s = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        if np.unique(yt).size < 2:
            continue
        f1s.append(f1_score(yt, y_pred_bin[:, c], zero_division=0))
    return float(np.mean(f1s)) if f1s else float("nan")


def micro_f1(y_true: np.ndarray, y_pred_bin: np.ndarray) -> float:
    if y_true.size == 0:
        return float("nan")
    return float(f1_score(y_true.ravel(), y_pred_bin.ravel(), zero_division=0))


def violation_rate(pred_bin: np.ndarray, m_parent: np.ndarray) -> float:
    child, parent = np.where(m_parent == 1)
    if len(child) == 0:
        return 0.0
    child_on = pred_bin[:, child] == 1
    parent_off = pred_bin[:, parent] == 0
    num = int((child_on & parent_off).sum())
    den = int(child_on.sum())
    return float(num / den) if den > 0 else 0.0


def apply_closure(pred_bin: np.ndarray, m_ancestor: np.ndarray) -> np.ndarray:
    pred = pred_bin.copy()
    for child in range(pred.shape[1]):
        anc = np.where(m_ancestor[child] == 1)[0]
        if len(anc) == 0:
            continue
        rows = pred[:, child] == 1
        pred[np.ix_(rows, anc)] = 1
    return pred


EPOCHS = 100
LAMBDA_H = 0.2
EARLY_STOPPING_PATIENCE = 10
EARLY_STOPPING_MIN_DELTA = 1e-4
best_state = None
best_metric = -1.0
best_epoch = 0
bad_epochs = 0
history = []

y_valid_true = collect_targets(valid_loader)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    total_n = 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        logits = model(batch.x, batch.edge_index, batch.batch)
        yb = reshape_batch_targets(batch.y, n_classes=500)
        loss_b = bce_loss(logits, yb) + SOFT_F1_ALPHA * soft_f1_loss(logits, yb)
        loss_h = hierarchy_penalty(logits)
        loss = loss_b + LAMBDA_H * loss_h
        loss.backward()
        optimizer.step()

        n = yb.shape[0]
        total_loss += float(loss.item()) * n
        total_n += n

    train_loss = total_loss / max(total_n, 1)
    val_logits = predict_logits(model, valid_loader)
    val_probs = 1.0 / (1.0 + np.exp(-val_logits))
    val_macro = macro_ap(y_valid_true, val_probs)
    val_micro = micro_ap(y_valid_true, val_probs)

    pred_bin = (val_probs >= 0.5).astype(np.uint8)
    val_macro_f1 = macro_f1(y_valid_true, pred_bin)
    val_micro_f1 = micro_f1(y_valid_true, pred_bin)
    v_before = violation_rate(pred_bin, M_parent_np)
    pred_closed = apply_closure(pred_bin, M_ancestor_np)
    v_after = violation_rate(pred_closed, M_parent_np)

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_macro_ap": val_macro,
            "val_micro_ap": val_micro,
            "val_macro_f1": val_macro_f1,
            "val_micro_f1": val_micro_f1,
            "violation_before": v_before,
            "violation_after": v_after,
        }
    )

    if np.isfinite(val_macro_f1) and val_macro_f1 > (best_metric + EARLY_STOPPING_MIN_DELTA):
        best_metric = val_macro_f1
        best_epoch = epoch
        bad_epochs = 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1

    print(
        f"Epoch {epoch:02d} | loss={train_loss:.4f} | "
        f"val_macro_ap={val_macro:.4f} | val_micro_ap={val_micro:.4f} | "
        f"val_macro_f1={val_macro_f1:.4f} | val_micro_f1={val_micro_f1:.4f} | "
        f"viol={v_before:.4f}->{v_after:.4f}"
    )

    if bad_epochs >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping: brak poprawy val_macro_f1 przez {EARLY_STOPPING_PATIENCE} epok. "
            f"Best epoch={best_epoch}, best_val_macro_f1={best_metric:.4f}"
        )
        break

if best_state is not None:
    model.load_state_dict(best_state)
    print("Wczytano najlepszy checkpoint z val_macro_f1 =", best_metric, "(epoka", best_epoch, ")")

Epoch 01 | loss=0.3366 | val_macro_ap=0.1134 | val_micro_ap=0.6419 | val_macro_f1=0.0589 | val_micro_f1=0.6014 | viol=0.0304->0.0000
Epoch 02 | loss=0.2699 | val_macro_ap=0.1516 | val_micro_ap=0.6276 | val_macro_f1=0.0905 | val_micro_f1=0.6226 | viol=0.0264->0.0000


C:\Users\ratch\AppData\Local\Temp\ipykernel_44680\428372469.py:122: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


Epoch 03 | loss=0.2526 | val_macro_ap=0.1928 | val_micro_ap=0.6469 | val_macro_f1=0.1400 | val_micro_f1=0.6767 | viol=0.0449->0.0000


C:\Users\ratch\AppData\Local\Temp\ipykernel_44680\428372469.py:122: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


Epoch 04 | loss=0.2411 | val_macro_ap=0.2081 | val_micro_ap=0.6235 | val_macro_f1=0.1399 | val_micro_f1=0.6517 | viol=0.0436->0.0000


...

Epoch 48 | loss=0.1739 | val_macro_ap=0.3390 | val_micro_ap=0.6237 | val_macro_f1=0.3968 | val_micro_f1=0.7256 | viol=0.0946->0.0000


C:\Users\ratch\AppData\Local\Temp\ipykernel_44680\428372469.py:122: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


Epoch 49 | loss=0.1738 | val_macro_ap=0.3485 | val_micro_ap=0.6287 | val_macro_f1=0.4084 | val_micro_f1=0.7266 | viol=0.1027->0.0000


C:\Users\ratch\AppData\Local\Temp\ipykernel_44680\428372469.py:122: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


Epoch 50 | loss=0.1736 | val_macro_ap=0.3564 | val_micro_ap=0.6346 | val_macro_f1=0.4179 | val_micro_f1=0.7319 | viol=0.0962->0.0000
Early stopping: brak poprawy val_macro_f1 przez 10 epok. Best epoch=40, best_val_macro_f1=0.4259
Wczytano najlepszy checkpoint z val_macro_f1 = 0.42585940705668496 (epoka 40 )


In [10]:
# Metryki per-poziom (0..17)
val_logits = predict_logits(model, valid_loader)
val_probs = 1.0 / (1.0 + np.exp(-val_logits))
y_true = collect_targets(valid_loader)

class_to_idx = {c: i for i, c in enumerate(class_cols)}
level_to_cols = defaultdict(list)
for cls, lvl in class_levels.items():
    if cls in class_to_idx:
        level_to_cols[lvl].append(class_to_idx[cls])

rows = []
for lvl in range(18):
    cols = level_to_cols.get(lvl, [])
    if len(cols) == 0:
        rows.append({"level": lvl, "n_classes": 0, "macro_ap": np.nan, "micro_ap": np.nan})
        continue
    yt = y_true[:, cols]
    ys = val_probs[:, cols]
    rows.append(
        {
            "level": lvl,
            "n_classes": len(cols),
            "macro_ap": macro_ap(yt, ys),
            "micro_ap": micro_ap(yt, ys),
        }
    )

level_metrics = pd.DataFrame(rows)
level_metrics

C:\Users\ratch\AppData\Local\Temp\ipykernel_44680\1978517130.py:3: RuntimeWarning: overflow encountered in exp
  val_probs = 1.0 / (1.0 + np.exp(-val_logits))


,level,n_classes,macro_ap,micro_ap
0,0,1,NaN,1.000000
1,1,3,0.332783,0.955520
2,2,8,0.410974,0.833602
3,3,11,0.568877,0.830412
4,4,15,0.538104,0.732984
5,5,25,0.430235,0.645269
6,6,29,0.468535,0.684974
7,7,30,0.513962,0.673058
8,8,33,0.441811,0.579079
9,9,74,0.340754,0.465573


In [11]:
# Zapis artefaktow etapu 3
OUT_DIR = DATA_DIR / "stage3_artifacts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ckpt_path = OUT_DIR / "hier_gnn_best.pt"
hist_path = OUT_DIR / "training_history.json"
level_path = OUT_DIR / "level_metrics.csv"
valid_pred_path = OUT_DIR / "valid_predictions.npz"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "seed": SEED,
        "class_columns": class_cols,
        "class_levels": class_levels,
        "lambda_h": LAMBDA_H,
    },
    ckpt_path,
)

with hist_path.open("w", encoding="utf-8") as f:
    json.dump(history, f, indent=2)

level_metrics.to_csv(level_path, index=False)

val_pred_bin = (val_probs >= 0.5).astype(np.uint8)
val_pred_closed = apply_closure(val_pred_bin, M_ancestor_np)
np.savez_compressed(
    valid_pred_path,
    y_true=y_true,
    y_prob=val_probs,
    y_pred_bin=val_pred_bin,
    y_pred_closed=val_pred_closed,
)

print("Zapisano:")
print("-", ckpt_path)
print("-", hist_path)
print("-", level_path)
print("-", valid_pred_path)

Zapisano:
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_artifacts\hier_gnn_best.pt
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_artifacts\training_history.json
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_artifacts\level_metrics.csv
- c:\Users\ratch\Desktop\hackathon_2026\ensemble2026\1_ontology\data\stage3_artifacts\valid_predictions.npz
